# Server-side classification evaluation (template)

This notebook is a **template** for comparing models on the `ClassificationCategory` / `ClassificationModel` stack.

**Avoid running model cells on a laptop** if you execute elsewhere: `HF_LLM`, `Ollama_LLM`, etc. **download weights** or open network services. Set `enabled=True` only in `MODEL_SPECS` on the machine that should load models.

Categories are read from JSON under `src/nps_crawling/classification/configurations/categories` (layout follows `Config.CLASSIFICATION_CONFIG_USE_NAME_FILES`).


In [ ]:
%pip install scikit-learn
%pip install pandas
%pip install -e ..
%pip install accelerate
%pip install transformers
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

: 

## 1. Discover categories from configuration files

`discover_categories()` loads each category JSON from the packaged tree (skips `Default` / `default`).

Set `CATEGORY_CONFIG_PATHS` to a non-empty list of `Path` objects to load **only** those files (absolute paths or paths relative to the notebook cwd).


In [3]:
from __future__ import annotations

import json

from nps_crawling.config import Config
from nps_crawling.classification.categories.category import ClassificationCategory
import json
from nps_crawling.config import Config

nps_category_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS Category" / "c0b1409c1550fcafe2a84875f32bb495385beac0eccf6d09d7e9eac4c266c1f7.json"
nps_value_category_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS Value Category" / "12b0b9acf27aee693b1d518d68a4b6e9481a53fb55245ab4f08e7c50274f433b.json"
nps_numeric_nps_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "Has Numeric NPS" / "ee63ed22edbeb29b02a976cfdd209f172cff421b5bde2009e1a0c0193f83a38f.json"
nps_all_file = Config.ROOT_DIR / "src" / "nps_crawling" / "classification" / "configurations" / "categories" / "NPS All" / "971f154bd74fd43de50c238df15f9721f97ad9114a51cac2f5d5a3f5767a68cd.json"

with open(nps_category_file, "r") as f:
    data = json.load(f)
    nps_category = ClassificationCategory.from_dict(data)
with open(nps_value_category_file, "r") as f:
    data = json.load(f)
    nps_value_category = ClassificationCategory.from_dict(data)
with open(nps_numeric_nps_file, "r") as f:
    data = json.load(f)
    has_numeric_nps = ClassificationCategory.from_dict(data)
with open(nps_all_file, "r") as f:
    data = json.load(f)
    nps_all = ClassificationCategory.from_dict(data)

## 2. Model instances (edit `MODEL_SPECS` on the runner machine)

**Option A — `get_model`:** use `ClassificationModelName` + `get_model(...)` (kwargs forwarded to the concrete class).

**Option B — direct classes:** import `HF_LLM`, `OpenAIModel`, `Ollama_LLM`, … and construct them explicitly (same as the registry does internally).

Set `enabled=True` for specs you want `build_models()` to instantiate.


In [25]:
import os
with open(r"C:\Users\silas\openai.txt", "r") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key

In [16]:
from nps_crawling.classification.models.registry import ClassificationModelName, get_model
from nps_crawling.classification.models.model import ClassificationModel
model = get_model(ClassificationModelName.BGE_BASE, "sentence-transformers/paraphrase-multilingual-mpnet-base-v2", examples="handselected")

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

c:\Users\silas\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\cache\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from nps_crawling.classification.model_pipeline import ClassificationModelPipeline

#pipeline = ClassificationModelPipeline("GPT_All", classification_configuration={nps_all:model})

## 3. Evaluation grid

`ClassificationModel.evaluate(category, text_column, test_size=None)` reads the ground-truth CSV from `category.csv_path` (relative paths are anchored at `Config.ROOT_DIR` in the library). Metrics are written back to each model's JSON config under `evaluation_results`.


In [17]:
model.train(nps_category)
model.train(has_numeric_nps)

In [18]:
model.evaluate(nps_category)
model.evaluate(has_numeric_nps)

c:\Users\silas\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\silas\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\silas\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'has_numeric_nps': {'0': {'precision': 0.679144385026738,
   'recall': 0.6939890710382514,
   'f1-score': 0.6864864864864865,
   'support': 183.0},
  '1': {'precision': 0.654320987654321,
   'recall': 0.6385542168674698,
   'f1-score': 0.6463414634146342,
   'support': 166.0},
  'accuracy': 0.667621776504298,
  'macro avg': {'precision': 0.6667326863405295,
   'recall': 0.6662716439528606,
   'f1-score': 0.6664139749505603,
   'support': 349.0},
  'weighted avg': {'precision': 0.667337267651892,
   'recall': 0.667621776504298,
   'f1-score': 0.6673917190654908,
   'support': 349.0}},
 'time_per_snippet': 0.046441787965616044}